In [1]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC_Stable_logdomain_fixed import NESTotalAnsatz, create_machine,\
    ha,SingleStateAnsatz,create_single_machine,\
        Ham_psi,Ham_Psi,NES_loss_energy_stable,nes_vmc_gradient,hi,E_fcis,\
        NESFermionHopRule,compute_qgt,sampler_info,create_machine_aux
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time

# ========== 你原有全局参数（直接复用） ==========
# 单系统希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
K = 3  # NES 扩展副本数
hi_ext = hi ** K  # 扩展希尔伯特空间
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
single_edges = ((0, 1), (2, 3))  # 费米子跃迁边
g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)

total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_machine_aux, total_graphdef,total_params = create_machine_aux(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: log(ψ) ∈ ℜ → ψ = exp(logψ) ∈ ℜ₊ (real NN gives only positive wave-function).

INFO:NES_VMC_Stable_logdomain_fixed:初始化 H₂ 分子和 FCI 基准能量
INFO:NES_VMC_Stable_logdomain_fixed:FCI 基准能量: E0=-1.01546825, E1=-0.87542794, E2=-0.42938376


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


DEBUG:NES_VMC_Stable_logdomain_fixed:Hamiltonian 算符创建完成, hilbert space size: 4
INFO:NES_VMC_Stable_logdomain_fixed:Hilbert space: 4 个自旋轨道
INFO:NES_VMC_Stable_logdomain_fixed:扩展 Hilbert space (K=3): 12 个自旋轨道
DEBUG:NES_VMC_Stable_logdomain_fixed:单粒子跃迁边: ((0, 1), (2, 3))
DEBUG:NES_VMC_Stable_logdomain_fixed:SingleStateAnsatz 初始化: n_spin=4, hidden=12
DEBUG:NES_VMC_Stable_logdomain_fixed:SingleStateAnsatz 初始化: n_spin=4, hidden=12
DEBUG:NES_VMC_Stable_logdomain_fixed:SingleStateAnsatz 初始化: n_spin=4, hidden=12
INFO:NES_VMC_Stable_logdomain_fixed:创建 single machine
INFO:NES_VMC_Stable_logdomain_fixed:创建 single machine
INFO:NES_VMC_Stable_logdomain_fixed:创建 single machine


In [2]:
import logging
# 日志配置
logger = logging.getLogger('NES_VMC')
logger.setLevel(logging.DEBUG)
# 阻止日志向上传播
logger.propagate = False
# 清除所有旧handler，防止重复打印
logger.handlers.clear()

# 自定义日志格式：只打印内容，不带等级、logger名
simple_formatter = logging.Formatter("%(message)s", datefmt="%H:%M:%S")

# 1. 文件输出处理器
file_handler = logging.FileHandler("nes_vmc_0616_K3.log", mode="w", encoding="utf-8")
file_handler.setFormatter(simple_formatter)
file_handler.setLevel(logging.DEBUG)
logger.addHandler(file_handler)

print('库导入完成')

库导入完成


In [3]:
logger.info("=" * 60)
logger.info("NES-VMC 训练开始")
logger.info("=" * 60)

# 参数设置
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER = 400
Natural_Grad = True
SINGLE_SIZE = hi.size

logger.info(f"参数: K={K}, N_CHAINS={N_CHAINS}, N_SAMPLES_PER_CHAIN={N_SAMPLES_PER_CHAIN}")
logger.info(f"      N_ITER={N_ITER}, Natural_Grad={Natural_Grad}, SWEEP_SIZE={SWEEP_SIZE}")

# 创建 Ansatz
logger.info("创建 NESTotalAnsatz...")
total_ansatz = NESTotalAnsatz(4, K, 12, rngs=nnx.Rngs(11))
logger.info("Ansatz 创建完成")

# 创建各种 wrapper
logger.info("创建 wrapper functions...")
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_aux_machine, total_aux_graphdef, total_aux_params = create_machine_aux(total_ansatz)


single_machine_list = []
for i, ansatz in enumerate(total_ansatz.single_ansatz_list):
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    logger.debug(f"  创建 single_machine[{i}]")

logger.info(f"共创建 {len(single_machine_list)} 个 single_machine")

# 优化器
optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)
logger.info("优化器初始化完成 (SGD, lr=0.01)")

# 构造扩展边
ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)
logger.info(f"扩展跃迁边: {ext_edges.tolist()}")

# 创建采样器
nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE
)
logger.info(f"采样器创建完成: {N_CHAINS} chains, sweep_size={SWEEP_SIZE}")

# 采样器状态初始化
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)
logger.info("采样器状态初始化完成")

# 训练循环
print("\n" + "="*60)
print("开始多链 NES-VMC 训练")
print("="*60)
print(f"基态能量={E_fcis[0]:.8f} Ha | 1st激发态={E_fcis[1]:.8f} Ha | 2st激发态={E_fcis[2]:.8f} Ha")

history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix': [],
    'grad_norm': [],
    'samples': [],
    'log_Psi_mean': [],
    'log_Psi_min': [],
    'log_Psi_max': [],
}

start_time = time.time()

for step in range(N_ITER):
    # 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN
    )
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, 4)
    # 计算梯度
    grad, loss_mean, E_L_mean = nes_vmc_gradient(
        ha=ha,
        total_aux_machine=total_aux_machine,
        total_machine=total_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch)

    grad_flat, grad_unravel_fn = ravel_pytree(grad)
    grad_norm = jnp.linalg.norm(grad_flat)

    # 自然梯度
    if Natural_Grad:
        qgt_reg, unravel_fn = compute_qgt(
            total_machine, total_params, x_batch, diag_shift=0.001
        )
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        natural_grad = grad_unravel_fn(natural_grad_flat)
        grad = natural_grad

    # 参数更新
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)

    # 计算监控量
    log_Psi_batch = total_machine(total_params, x_batch)
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)

    # 记录历史
    history['step'].append(step)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(jnp.real(log_Psi_batch).mean())
    history['log_Psi_min'].append(jnp.real(log_Psi_batch).min())
    history['log_Psi_max'].append(jnp.real(log_Psi_batch).max())
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)

    # 打印
    if step % 20 == 0 or step == N_ITER - 1:
        logger.info(f"\n---------------- Step {step} ---------------------")
        logger.info(f"log_Psi(real): mean={jnp.real(log_Psi_batch).mean():.3f} | "
                f"min={jnp.real(log_Psi_batch).min():.3f} | "
                f"max={jnp.real(log_Psi_batch).max():.3f}")
        
        logger.info(f"grad norm = {grad_norm:.4f}")
        logger.info(f"Loss: {loss_mean:.6f}")
        logger.info(f"E0={eig_vals[0]:.8f} Ha | E1={eig_vals[1]:.8f} Ha | E2={eig_vals[2]:.8f} Ha")
        logger.info("#" + "-"*58 + "#")

end_time = time.time()
logger.info(f"\n训练完成! 耗时: {end_time - start_time:.2f} 秒")

# 最终结果
logger.info("\n" + "="*60)
logger.info("最终结果")
logger.info("="*60)
final_E0 = history['energy_0st'][-1]
final_E1 = history['energy_1st'][-1]

logger.info(f"NES-VMC: E0={final_E0:.8f} Ha, E1={final_E1:.8f} Ha")
logger.info(f"FCI:     E0={E_fcis[0]:.8f} Ha, E1={E_fcis[1]:.8f} Ha")
logger.info(f"误差:    dE0={abs(final_E0-E_fcis[0]):.2e}, dE1={abs(final_E1-E_fcis[1]):.2e}")
logger.info("="*60)


DEBUG:NES_VMC_Stable_logdomain_fixed:SingleStateAnsatz 初始化: n_spin=4, hidden=12
DEBUG:NES_VMC_Stable_logdomain_fixed:SingleStateAnsatz 初始化: n_spin=4, hidden=12
DEBUG:NES_VMC_Stable_logdomain_fixed:SingleStateAnsatz 初始化: n_spin=4, hidden=12
INFO:NES_VMC_Stable_logdomain_fixed:创建 single machine
INFO:NES_VMC_Stable_logdomain_fixed:创建 single machine
INFO:NES_VMC_Stable_logdomain_fixed:创建 single machine



开始多链 NES-VMC 训练
基态能量=-1.01546825 Ha | 1st激发态=-0.87542794 Ha | 2st激发态=-0.42938376 Ha


KeyboardInterrupt: 